# Adding rows to Prophet embeddings using the published StandardScaler objects

This notebook shows how to:

- Download the cell line and intervention `StandardScaler` objects from the Hugging Face dataset repo
- Use them to transform new (unscaled) embeddings into the same scaled space as Prophet expects
- Create dummy example embeddings (random numbers) that you should replace with your own

Important:

- These scalers only ensure your vectors are on the same mean/variance scale.
- You must generate the unscaled embeddings using the same upstream pipeline that was used to create the original embeddings (same features, same order, same preprocessing). Otherwise, scaling alone will not make them compatible.

Files on Hugging Face (dataset repo `theislab/Prophet`):

- `embeddings/scalers/cell_line_standard_scaler.pkl`
- `embeddings/scalers/intervention_standard_scaler.pkl`


## 1. Imports and configuration

Set `REPO_ID` to the dataset repo containing the scalers.

If needed:

```bash
pip install huggingface_hub joblib scikit-learn
```

In [ ]:
from huggingface_hub import hf_hub_download
from joblib import load
import numpy as np
import pandas as pd

REPO_ID = "theislab/Prophet"
REPO_TYPE = "dataset"

np.random.seed(0)

## 2. Download and load the scalers

Use `hf_hub_download` with a `filename` inside the dataset repo (do not use the `blob/...` URL).

In [ ]:
cell_scaler_file = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename="embeddings/scalers/cell_line_standard_scaler.pkl",
)
iv_scaler_file = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename="embeddings/scalers/intervention_standard_scaler.pkl",
)

cell_scaler = load(cell_scaler_file)
iv_scaler = load(iv_scaler_file)

print("Cell line scaler n_features:", int(cell_scaler.mean_.shape[0]))
print("Intervention scaler n_features:", int(iv_scaler.mean_.shape[0]))

## 3. Cell line embeddings example

This example downloads the existing scaled cell line embedding table from Hugging Face, creates a single new unscaled 300-dimensional vector (random placeholder), scales it with the published scaler, appends it, and saves an updated CSV.

Replace the random vector with your own unscaled 300-dimensional cell line embedding produced by the same upstream pipeline used for Prophet.

In [ ]:
cell_emb_file = hf_hub_download(
    repo_id=REPO_ID,
    repo_type=REPO_TYPE,
    filename="embeddings/cell_line_embeddings/cell_line_embedding_full_ccle_300_scaled.csv",
)
cell_emb_scaled = pd.read_csv(cell_emb_file, index_col=0)

print("Loaded scaled cell line embeddings:")
print("  shape:", cell_emb_scaled.shape)
print("  first index entries:", list(cell_emb_scaled.index[:3]))
print("  first columns:", list(cell_emb_scaled.columns[:5]))

In [ ]:
new_cell_line_name = "my_new_cell_line"

n_cl_features = int(cell_scaler.mean_.shape[0])
new_cl_unscaled = np.random.normal(loc=0.0, scale=1.0, size=(1, n_cl_features))
new_cl_scaled = cell_scaler.transform(new_cl_unscaled)

new_cl_scaled_df = pd.DataFrame(
    new_cl_scaled,
    index=[new_cell_line_name],
    columns=cell_emb_scaled.columns,
)

cell_emb_scaled_updated = pd.concat([cell_emb_scaled, new_cl_scaled_df], axis=0)

out_cell_emb_path = "./cell_line_embedding_full_ccle_300_scaled_with_custom_row.csv"
cell_emb_scaled_updated.to_csv(out_cell_emb_path)

print("Added new row:", new_cell_line_name)
print("Updated shape:", cell_emb_scaled_updated.shape)
print("Wrote:", out_cell_emb_path)

## 4. Intervention (perturbation) embeddings example

The official intervention embedding table on Hugging Face is very large. This example creates a small "existing" embedding table with the correct dimensionality, adds one new intervention embedding (random placeholder), and saves a CSV.

Replace the random vector with your own unscaled intervention embedding produced by the same upstream pipeline used for Prophet.

Note: Prophet lowercases intervention identifiers internally, so prefer lowercased identifiers.

In [ ]:
n_iv_features = int(iv_scaler.mean_.shape[0])

existing_iv_names = [
    "existing_iv_a",
    "existing_iv_b",
    "negative_gene",
    "negative_drug",
]

existing_iv_types = [
    "gene",
    "drug",
    "gene",
    "drug",
]

existing_iv_unscaled = np.random.normal(loc=0.0, scale=1.0, size=(len(existing_iv_names), n_iv_features))
existing_iv_scaled = iv_scaler.transform(existing_iv_unscaled)

existing_iv_scaled_df = pd.DataFrame(
    existing_iv_scaled,
    index=[s.lower() for s in existing_iv_names],
    columns=[str(i) for i in range(n_iv_features)],
)
existing_iv_scaled_df.insert(0, "type", existing_iv_types)

new_iv_name = "my_new_iv"
new_iv_type = "drug"
new_iv_unscaled = np.random.normal(loc=0.0, scale=1.0, size=(1, n_iv_features))
new_iv_scaled = iv_scaler.transform(new_iv_unscaled)

new_iv_scaled_df = pd.DataFrame(
    new_iv_scaled,
    index=[new_iv_name.lower()],
    columns=[str(i) for i in range(n_iv_features)],
)
new_iv_scaled_df.insert(0, "type", [new_iv_type])

iv_emb_scaled_updated = pd.concat([existing_iv_scaled_df, new_iv_scaled_df], axis=0)

out_iv_emb_path = "./intervention_embeddings_scaled_with_custom_row.csv"
iv_emb_scaled_updated.to_csv(out_iv_emb_path)

print("Added new intervention:", new_iv_name.lower())
print("Updated shape:", iv_emb_scaled_updated.shape)
print("Wrote:", out_iv_emb_path)

## 5. Using the updated embeddings with Prophet

Once you have updated CSVs, you can point a Prophet model at them.

This example downloads a pretrained model checkpoint using `Prophet.from_pretrained`, then creates a new `Prophet` instance with your updated embedding files.

If you do not need both custom tables, you can pass only one custom path and keep the other from the pretrained download.

In [ ]:
from prophet import Prophet

base = Prophet.from_pretrained("base")

custom_model = Prophet(
    iv_emb_path=out_iv_emb_path,
    cl_emb_path=out_cell_emb_path,
    ph_emb_path=None,
    model_pth=base.model_pth,
)

print("Custom Prophet instance created.")
print("  model checkpoint:", custom_model.model_pth)
print("  iv embeddings:", out_iv_emb_path)
print("  cl embeddings:", out_cell_emb_path)

## 6. Next: replace the random placeholders

Replace:

- `new_cl_unscaled` with a real unscaled 300-dimensional cell line embedding
- `new_iv_unscaled` with a real unscaled intervention embedding

Make sure:

- The dimensionality matches what the scaler expects
- The upstream embedding pipeline matches the one used to generate Prophet embeddings
- Your intervention identifiers are lowercased (recommended)


## 7. Using your own embeddings

We encourage you to go beyond the default embeddings provided with Prophet and experiment with your own. Prophet is designed to be modular: any embedding that captures meaningful biological signal for cell lines or interventions can potentially improve predictions for your specific use case.

For example, [PRESAGE (Littman et al., 2025)](https://www.biorxiv.org/content/10.1101/2025.06.03.657653v1) systematically explores a large variety of gene embedding sources and shows that knowledge source selection has a strong impact on perturbation response prediction. The embeddings evaluated there — and others you may derive from your own data — are all candidates for use with Prophet via the workflow shown in this notebook.